In [2]:
import pandas as pd
import numpy as np
import torch 
from torch_geometric.data import Data
import random

/home/ishido/Desktop/Aj/New proj/HELP extend/environment/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
ppi = pd.read_csv("../../../data/kidney/ppi.csv")
labels = pd.read_csv("../../../data/kidney/labels.csv")
expressions = pd.read_csv("../../../data/kidney/expression.csv")
mutations = pd.read_csv("../../../data/kidney/mutations.csv")
model = pd.read_csv("../../../data/raw/depmap/Model.csv")
raw_label = pd.read_csv("../../../data/raw/depmap/CRISPRGeneEffect.csv")

In [7]:
raw_label

,Unnamed: 0,A1BG (1),A1CF (29974),A2M (2),A2ML1 (144568),A3GALT2 (127550),A4GALT (53947),A4GNT (51146),AAAS (8086),AACS (65985),...,ZWILCH (55055),ZWINT (11130),ZXDA (7789),ZXDB (158586),ZXDC (79364),ZYG11A (440590),ZYG11B (79699),ZYX (7791),ZZEF1 (23140),ZZZ3 (26009)
0,ACH-000015,-0.101628,0.033423,-0.084998,0.084588,-0.025752,0.014998,0.127182,-0.309637,0.087638,...,-0.329088,-0.271706,0.008982,0.066563,0.130267,0.044343,-0.182709,-0.107747,-0.135141,-0.236502
1,ACH-000045,0.071876,0.096220,0.030159,0.128536,0.010489,-0.062870,-0.067315,-0.472265,-0.116420,...,-0.149970,-0.610225,0.075264,0.227191,0.011966,-0.065919,-0.171262,0.023059,-0.206351,-0.458801
2,ACH-000094,0.100488,0.204729,0.111304,0.220877,0.022709,-0.009274,0.086337,-0.224036,0.082950,...,0.014893,-0.882753,-0.238187,0.024990,-0.128601,0.359461,-0.230729,0.231520,0.042054,-0.762764
3,ACH-000096,0.182435,-0.004316,-0.117120,0.135999,-0.169988,-0.200839,0.203096,-0.027621,0.013396,...,-0.312683,-0.410344,0.199127,0.067745,0.088566,0.175735,-0.163985,-0.366537,-0.122897,-0.104066
4,ACH-000099,-0.103127,0.075134,-0.071087,0.036089,-0.016140,-0.154740,0.024042,0.045064,-0.018520,...,-0.318896,-0.603687,0.034780,0.109159,0.150058,-0.031689,-0.422116,0.127047,-0.238114,-0.257072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1181,ACH-002687,-0.063994,0.014906,0.119931,0.266804,0.012315,0.074110,0.006276,-0.374605,-0.065559,...,-0.262538,-0.878076,-0.022727,0.112161,0.163097,-0.017343,-0.267976,-0.140125,-0.033730,-0.738082
1182,ACH-002708,-0.008109,0.051431,0.048292,0.254607,-0.027347,-0.178628,0.223170,-0.797440,-0.155090,...,0.049700,-0.635168,-0.190640,0.138895,0.114770,-0.099998,-0.101318,-0.233697,0.040225,-0.449273
1183,ACH-002782,-0.019900,-0.034184,0.065603,0.128023,0.025150,-0.118020,0.021829,-0.324454,0.082824,...,0.005114,-0.270304,0.015133,0.045545,0.012244,-0.025254,-0.271036,-0.067795,-0.086425,-0.087778
1184,ACH-002785,-0.063686,-0.039207,0.025933,0.102223,-0.170395,0.061861,-0.044682,-0.347906,0.047099,...,0.081239,-0.157048,-0.024652,0.024075,-0.080117,-0.081808,-0.269077,-0.019338,-0.235562,-0.115628


In [17]:
raw_label.rename(columns={"Unnamed: 0": "ModelID"}, inplace=True)
global_mean = raw_label.drop("ModelID", axis=1).mean(axis=0)
global_mean.index = global_mean.index.str.split(" (", regex=False).str[0]





In [18]:
global_mean

A1BG      -0.003560
A1CF      -0.013516
A2M        0.010347
A2ML1      0.101075
A3GALT2   -0.080971
             ...   
ZYG11A    -0.010723
ZYG11B    -0.179038
ZYX        0.027714
ZZEF1     -0.141770
ZZZ3      -0.306917
Length: 18435, dtype: float64

In [11]:
ppi_genes = pd.concat([ppi['A'], ppi['B']]).unique()
gene_to_idx = {gene: i for i, gene in enumerate(ppi_genes)}

In [12]:
edge_src = ppi['A'].map(gene_to_idx).values
edge_dst = ppi["B"].map(gene_to_idx).values

In [13]:
edge_index = torch.tensor([edge_src,edge_dst],dtype=torch.long)
edge_index.shape

/tmp/ipykernel_17800/3632093261.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  edge_index = torch.tensor([edge_src,edge_dst],dtype=torch.long)


torch.Size([2, 1528076])

In [14]:
edge_attr = torch.tensor(ppi[['combined_score', 'coexpression']].values, dtype=torch.float)
edge_attr = (edge_attr - edge_attr.mean(dim=0)) / edge_attr.std(dim=0)
edge_attr.shape

torch.Size([1528076, 2])

In [19]:
expr_no_id = expressions.drop('ModelID', axis=1)
mut_no_id = mutations.drop('ModelID', axis=1)
mutations_aligned = mut_no_id.reindex(columns=ppi_genes, fill_value=0)
labels_no_id = labels.drop('ModelID', axis=1)
labels_reindexed = labels_no_id.reindex(columns=ppi_genes)
labels_reindexed = labels_reindexed - global_mean


In [20]:
# VHL mutation?
vhl_mutation = mutations[['ModelID']].copy()
vhl_mutation['VHL_mutated'] = mutations_aligned['VHL'].values

# example
hif1a_diff = labels_reindexed['HIF1A'].values

for i in range(len(labels)):
    print(f"{labels['ModelID'].iloc[i]}  VHL_mut: {int(vhl_mutation['VHL_mutated'].iloc[i])}  HIF1A_diff: {hif1a_diff[i]:.4f}")


ACH-000096  VHL_mut: 2  HIF1A_diff: 0.2595
ACH-000262  VHL_mut: 2  HIF1A_diff: 0.2044
ACH-000375  VHL_mut: 0  HIF1A_diff: -0.1202
ACH-000411  VHL_mut: 0  HIF1A_diff: 0.2180
ACH-000533  VHL_mut: 2  HIF1A_diff: 0.1677
ACH-000907  VHL_mut: 2  HIF1A_diff: 0.4823
ACH-001687  VHL_mut: 2  HIF1A_diff: 0.1756
ACH-000159  VHL_mut: 2  HIF1A_diff: 0.0577
ACH-000234  VHL_mut: 0  HIF1A_diff: 0.2143
ACH-001310  VHL_mut: 0  HIF1A_diff: -0.0492
ACH-001398  VHL_mut: 0  HIF1A_diff: 0.1693
ACH-000189  VHL_mut: 2  HIF1A_diff: 0.3015
ACH-000250  VHL_mut: 2  HIF1A_diff: -0.1108
ACH-000272  VHL_mut: 0  HIF1A_diff: -0.0438
ACH-000607  VHL_mut: 2  HIF1A_diff: 0.0972
ACH-001688  VHL_mut: 0  HIF1A_diff: 0.6798
ACH-000433  VHL_mut: 2  HIF1A_diff: 0.2611
ACH-000484  VHL_mut: 0  HIF1A_diff: -0.1744
ACH-001163  VHL_mut: 0  HIF1A_diff: 0.2316
ACH-000313  VHL_mut: 2  HIF1A_diff: 0.4131
ACH-000459  VHL_mut: 2  HIF1A_diff: 0.4577
ACH-001194  VHL_mut: 2  HIF1A_diff: 0.5502
ACH-001532  VHL_mut: 0  HIF1A_diff: -0.1364
ACH-0

In [21]:
print(f"HIF1A global mean: {global_mean['HIF1A']:.4f}")
print(f"PAX8 global mean: {global_mean['PAX8']:.4f}")


HIF1A global mean: 0.1288
PAX8 global mean: -0.1766


In [22]:
for i in range(len(labels)):
    print(f"{labels['ModelID'].iloc[i]}  PAX8_diff: {labels_reindexed['PAX8'].iloc[i]:.4f}")


ACH-000096  PAX8_diff: -0.1122
ACH-000262  PAX8_diff: -0.9903
ACH-000375  PAX8_diff: 0.0740
ACH-000411  PAX8_diff: -1.6971
ACH-000533  PAX8_diff: 0.0100
ACH-000907  PAX8_diff: -0.1196
ACH-001687  PAX8_diff: -1.4852
ACH-000159  PAX8_diff: -1.7242
ACH-000234  PAX8_diff: -0.2361
ACH-001310  PAX8_diff: -0.0976
ACH-001398  PAX8_diff: -0.4362
ACH-000189  PAX8_diff: -0.3269
ACH-000250  PAX8_diff: -1.2941
ACH-000272  PAX8_diff: -1.8449
ACH-000607  PAX8_diff: -0.0262
ACH-001688  PAX8_diff: -1.6768
ACH-000433  PAX8_diff: -0.7850
ACH-000484  PAX8_diff: -0.1913
ACH-001163  PAX8_diff: 0.0624
ACH-000313  PAX8_diff: -1.2016
ACH-000459  PAX8_diff: -0.4687
ACH-001194  PAX8_diff: -0.9560
ACH-001532  PAX8_diff: 0.1659
ACH-000709  PAX8_diff: -1.2980
ACH-002533  PAX8_diff: -1.2832
ACH-000385  PAX8_diff: 0.2455
ACH-000649  PAX8_diff: -1.5085
ACH-000792  PAX8_diff: -0.6719
ACH-000495  PAX8_diff: -0.7053
ACH-002528  PAX8_diff: -1.0945
ACH-000317  PAX8_diff: -1.5543
ACH-000684  PAX8_diff: -0.9660


In [ ]:
graph_list = []
for i in range(32):
    expr_row = expr_no_id[ppi_genes].iloc[i].values
    mut_row = mutations_aligned.iloc[i].values
    x = torch.tensor(np.column_stack([expr_row, mut_row]), dtype=torch.float)
    
    y_raw = labels_reindexed.iloc[i]
    cell_mask = torch.tensor((~y_raw.isna().values) & np.isin(ppi_genes, labels_no_id.columns), dtype=torch.bool)
    y = torch.tensor(y_raw.fillna(0).values, dtype=torch.float)
    
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, mask=cell_mask)
    graph_list.append(data)

In [ ]:
graph_list

In [ ]:
random.seed(12)
random.shuffle(graph_list)

train_data = graph_list[:26]
test_data = graph_list[26:]

from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1)

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GATConv


In [ ]:
class KidneyGAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.expr_encoder = nn.Linear(1, 32)
        self.mut_encoder = nn.Linear(1, 32)
        self.conv1 = GATConv(64, 64, heads=4, edge_dim=2)
        self.conv2 = GATConv(64 * 4, 64, heads=4, edge_dim=2, concat=False)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(64, 1)
    
    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        
        expr = self.expr_encoder(x[:, 0:1])
        mut = self.mut_encoder(x[:, 1:2])
        x = torch.cat([expr, mut], dim=1)
        
        x = self.conv1(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.out(x)
        return x.squeeze()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KidneyGAT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.MSELoss()

In [ ]:
epochs = 200
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        pred = model(batch)
        loss = loss_fn(pred[batch.mask], batch.y[batch.mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
test_losses = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch)
        loss = loss_fn(pred[batch.mask], batch.y[batch.mask])
        test_losses.append(loss.item())
print(f"Test MSE: {np.mean(test_losses):.4f}")

In [ ]:
from scipy.stats import pearsonr
correlations = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch)
        p = pred[batch.mask].cpu().numpy()
        a = batch.y[batch.mask].cpu().numpy()
        r, _ = pearsonr(p, a)
        correlations.append(r)
print(f"Test Pearson r: {np.mean(correlations):.4f}")

In [ ]:
model.eval()
sample = test_data[0].to(device)

with torch.no_grad():
    expr = model.expr_encoder(sample.x[:, 0:1])
    mut = model.mut_encoder(sample.x[:, 1:2])
    x = torch.cat([expr, mut], dim=1)
    
    x, attn1 = model.conv1(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)
    x = model.relu(x)
    x = model.dropout(x)
    
    x, attn2 = model.conv2(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)

edge_idx, scores = attn2
print(f"Edges: {edge_idx.shape}")
print(f"Attention scores: {scores.shape}")

In [ ]:
avg_attn = scores.mean(dim=1).cpu().numpy()
src = edge_idx[0].cpu().numpy()
dst = edge_idx[1].cpu().numpy()

mask = src != dst
src_filtered = src[mask]
dst_filtered = dst[mask]
attn_filtered = avg_attn[mask]

idx_to_gene = {i: gene for gene, i in gene_to_idx.items()}
print(f"Total edges (no self-loops): {len(attn_filtered)}")

In [ ]:
top_k = 20
top_indices = np.argsort(attn_filtered)[-top_k:][::-1]

print("Top 20 gene interactions by attention weight:\n")
for rank, i in enumerate(top_indices, 1):
    g1 = idx_to_gene[src_filtered[i]]
    g2 = idx_to_gene[dst_filtered[i]]
    print(f"{rank:2d}. {g1:10s} <-> {g2:10s}  attn: {attn_filtered[i]:.4f}")

In [ ]:
known_cancer_genes = ['VHL', 'TP53', 'EGFR', 'PTEN', 'MTOR', 'MYC', 'KRAS', 'BRCA1', 'PIK3CA', 'BRAF']

for gene in known_cancer_genes:
    if gene not in gene_to_idx:
        print(f"{gene} — not in PPI network")
        continue
    
    idx = gene_to_idx[gene]
    
    gene_mask = (src_filtered == idx) | (dst_filtered == idx)
    gene_attn = attn_filtered[gene_mask]
    gene_src = src_filtered[gene_mask]
    gene_dst = dst_filtered[gene_mask]
    
    if len(gene_attn) == 0:
        print(f"{gene} — no edges found")
        continue
    
    top5 = np.argsort(gene_attn)[-5:][::-1]
    
    print(f"\n{gene} — {len(gene_attn)} edges, max attn: {gene_attn.max():.4f}")
    for i in top5:
        partner = idx_to_gene[gene_dst[i]] if gene_src[i] == idx else idx_to_gene[gene_src[i]]
        print(f"  <-> {partner:12s}  attn: {gene_attn[i]:.4f}")